# 01 — Inject TOFU facts and run a first RMU unlearn

Colab-ready starter for the Lexsi Track 2b project **Unlearning vs. hiding**.

This notebook:

1. Installs dependencies (Python ≥ 3.12 required by SafeTune)
2. Logs into Hugging Face and loads `CohereLabs/tiny-aya-global` (gated, CC-BY-NC)
3. Loads TOFU `forget01` / `retain99`
4. Reports base-model forget-set accuracy (should be low — these facts are fictitious)
5. LoRA-fine-tunes so the model actually knows the facts
6. Verifies post-SFT forget/retain accuracy
7. Runs a first SafeTune `RMUTrainer` unlearn
8. Reports forget/retain accuracy after RMU

**Run this on Colab / Kaggle / HF ZeroGPU, not on the Mac.** Do not unlearn until injection is verified.

Primary method docs: https://safetune.lexsi.ai/user-guide/unlearn/rmu/

Default RMU `layer_id=7` is mid-early and **should be retuned on 36-layer Tiny Aya**. Record the layers you actually use.



## 1. Installs and repo path



In [ ]:
import sys
from pathlib import Path

print("Python", sys.version)
if sys.version_info < (3, 12):
    raise RuntimeError(
        "SafeTune requires Python >= 3.12 (https://safetune.lexsi.ai/getting-started/installation/). "
        "Change the Colab runtime to 3.12+ and rerun."
    )

# Locate the repo root (uploaded folder, Drive mount, or cwd).
CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/lexsi-unlearning"),
    Path("/content"),
]
ROOT = None
for cand in CANDIDATES:
    if (cand / "src" / "lexsi_unlearn").exists() or (cand / "configs" / "default.yaml").exists():
        ROOT = cand
        break
if ROOT is None:
    ROOT = Path.cwd()
    print("WARNING: could not find src/lexsi_unlearn; using cwd", ROOT)

import os
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT.resolve())

%pip install -q -r requirements.txt


## 2. Hugging Face login



In [ ]:
import os
from pathlib import Path
from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    from google.colab import userdata  # type: ignore
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if not token and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if line.startswith("HF_TOKEN=") and line.split("=", 1)[1].strip():
            token = line.split("=", 1)[1].strip().strip('"').strip("'")
            break

if token:
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face (token not printed).")
else:
    from huggingface_hub import notebook_login
    print("Accept the license first: https://huggingface.co/CohereLabs/tiny-aya-global")
    notebook_login()


## 3. Load config, Tiny Aya, and TOFU splits



In [ ]:
from pathlib import Path

from lexsi_unlearn.data import (
    build_direct_queries,
    load_config,
    load_tofu_split,
    qa_pairs,
)
from lexsi_unlearn.inject import attach_lora, load_base_model

cfg = load_config(ROOT / "configs" / "default.yaml")
MODEL_ID = cfg["model"]["id"]
print("Model:", MODEL_ID)
print("TOFU forget split:", cfg["tofu"]["forget_split"])
print("RMU layer_id (RETUNE on 36-layer Tiny Aya):", cfg["rmu"]["layer_id"])
print("RMU update_layers:", cfg["rmu"]["update_layers"])

model, tokenizer = load_base_model(MODEL_ID, dtype=cfg["model"].get("dtype", "bfloat16"))
print("Loaded base model.")

forget_ds = load_tofu_split(
    cfg["tofu"]["forget_split"],
    repo_id=cfg["tofu"]["repo_id"],
    max_examples=cfg["injection"].get("max_forget_examples"),
)
retain_ds = load_tofu_split(
    cfg["tofu"]["retain_split"],
    repo_id=cfg["tofu"]["repo_id"],
    max_examples=cfg["injection"].get("max_retain_examples"),
)
forget_pairs = qa_pairs(forget_ds)
retain_pairs = qa_pairs(retain_ds)
print(f"forget n={len(forget_pairs)}  retain (this run) n={len(retain_pairs)}")

# A compact eval slice so the first pass finishes in minutes. Raise later.
FORGET_EVAL = build_direct_queries(forget_pairs[:16])
RETAIN_EVAL = build_direct_queries(retain_pairs[:16])


## 4. Base-model accuracy (should be low on fictitious facts)

If the base model already "knows" forget-set authors, the ground truth is compromised — stop and inspect the examples before injecting.



In [ ]:
from lexsi_unlearn.evaluate import evaluate_model_on_items, summarize_forget_retain

MAX_NEW = int(cfg["eval"].get("max_new_tokens", 64))

base_forget = evaluate_model_on_items(model, tokenizer, FORGET_EVAL, max_new_tokens=MAX_NEW)
base_retain = evaluate_model_on_items(model, tokenizer, RETAIN_EVAL, max_new_tokens=MAX_NEW)
base_summary = summarize_forget_retain(forget=base_forget, retain=base_retain, label="base")
print(base_summary)
print("Sample base forget generations:")
for item, pred in zip(FORGET_EVAL[:3], base_forget["predictions"][:3]):
    print("--- Q:", item["prompt"])
    print("    gold:", item["target"])
    print("    gen :", pred)


## 5. LoRA injection fine-tune



In [ ]:
from lexsi_unlearn.inject import inject_facts

model = attach_lora(model, cfg["injection"]["lora"])
model.print_trainable_parameters()

# Weekend-sized LoRA SFT on forget01 + a compact retain slice.
model, adapter_dir = inject_facts(
    model,
    tokenizer,
    config=cfg,
    output_dir=ROOT / cfg["paths"]["inject_adapter_dir"],
)
print("Saved injection adapter to", adapter_dir)


## 6. Verify the model learned the facts (do not unlearn until this is high)



In [ ]:
inject_forget = evaluate_model_on_items(model, tokenizer, FORGET_EVAL, max_new_tokens=MAX_NEW)
inject_retain = evaluate_model_on_items(model, tokenizer, RETAIN_EVAL, max_new_tokens=MAX_NEW)
inject_summary = summarize_forget_retain(forget=inject_forget, retain=inject_retain, label="injected")
print("BASE   ", base_summary)
print("INJECT ", inject_summary)
print("Sample injected forget generations:")
for item, pred in zip(FORGET_EVAL[:3], inject_forget["predictions"][:3]):
    print("--- Q:", item["prompt"])
    print("    gold:", item["target"])
    print("    gen :", pred)

if inject_forget["substring_match"] < 0.5:
    raise RuntimeError(
        "Post-SFT forget-set accuracy is still low. Do not unlearn yet. "
        "Train longer, check the chat template, or shrink to a handful of distinctive facts."
    )


## 7. First RMU unlearn (SafeTune)

Confirmed API (https://safetune.lexsi.ai/user-guide/unlearn/rmu/):

```python
from safetune.runner import unlearn
trainer = unlearn.RMUTrainer(model)
unlearned = trainer.unlearn(forget=forget_batches, retain=retain_batches)
```

`layer_id=7` is the library default and is flagged for retuning on 36-layer Tiny Aya.



In [ ]:
from lexsi_unlearn.unlearn import (
    rmu_hparams_from_config,
    run_rmu,
    tokenize_unlearn_batches,
)

forget_batches = tokenize_unlearn_batches(tokenizer, forget_ds, batch_size=2)
retain_batches = tokenize_unlearn_batches(tokenizer, retain_ds, batch_size=2)
hparams = rmu_hparams_from_config(cfg)
print("RMU hparams (layer_id may need retuning):", hparams)

unlearned = run_rmu(
    model,
    forget_batches=forget_batches,
    retain_batches=retain_batches,
    hparams=hparams,
)

save_dir = ROOT / cfg["paths"]["unlearned_dir"]
save_dir.mkdir(parents=True, exist_ok=True)
# PEFT / HF save; if RMU returns a plain nn.Module this still works for HF models.
if hasattr(unlearned, "save_pretrained"):
    unlearned.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print("Saved unlearned model to", save_dir)
else:
    print("TODO: unlearned object has no save_pretrained:", type(unlearned))
    print("Inspect the return type of RMUTrainer.unlearn and save weights accordingly.")
    print("https://safetune.lexsi.ai/user-guide/unlearn/rmu/")


## 8. Forget / retain accuracy after RMU, vs. base and injected



In [ ]:
rmu_forget = evaluate_model_on_items(unlearned, tokenizer, FORGET_EVAL, max_new_tokens=MAX_NEW)
rmu_retain = evaluate_model_on_items(unlearned, tokenizer, RETAIN_EVAL, max_new_tokens=MAX_NEW)
rmu_summary = summarize_forget_retain(forget=rmu_forget, retain=rmu_retain, label="rmu")

print("=== before / after (direct questions, compact eval slice) ===")
for row in (base_summary, inject_summary, rmu_summary):
    print(row)

print("\nSample RMU forget generations:")
for item, pred in zip(FORGET_EVAL[:3], rmu_forget["predictions"][:3]):
    print("--- Q:", item["prompt"])
    print("    gold:", item["target"])
    print("    gen :", pred)

print()
print("Next (not in this notebook): paraphrases, format/translation/extraction prompts,")
print("layer-wise probes, logit lens, activation patching, CircuitKIT comparison, relearning.")
print("Do not claim success from a drop in direct-question accuracy alone.")
